In [ ]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(stringr)
library(scales)
library(patchwork)

In [ ]:
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/phenotype_files/aml_phenotype_occurrences.tsv .", intern=TRUE)

In [ ]:
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/phenotype_files/aml_concept_information.tsv .", intern=TRUE)

In [ ]:
data <- fread("aml_phenotype_occurrences.tsv")
concepts <- fread("aml_concept_information.tsv")

In [ ]:
library(ggplot2)

# Define base colors
base_colors <- c("#F7AA14FF", "#F5D000FF", "#F7E690FF", "#1BB6AFFF", "#088BBEFF", "#172869FF")

# Create 20 interpolated colors
palette_21 <- colorRampPalette(base_colors)(21)


In [ ]:
plot1 <- 
data %>% 
  group_by(concept_id) %>% 
  count() %>% 
  inner_join(concepts, by = "concept_id") %>% 
  arrange(n) %>% 
  mutate(concept_name = str_wrap(concept_name, width = 40)) %>% 
  ggplot(aes(x = reorder(concept_name, n), y = n, fill = concept_name)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  labs(
    x = "Concept Name",
    y = "Unique Electronic Health Record Count"
  ) +
  theme_minimal() +
  scale_fill_manual(values = palette_21) +
  theme(
    legend.title = element_text(size = 16), 
    legend.text  = element_text(size = 14),  
    axis.text.x = element_text(size = 20), 
    axis.title.x = element_text(size = 20), 
    axis.text.y = element_text(size = 20), 
    axis.title.y = element_text(size = 20)
  ) + 
  guides(fill = "none")

In [ ]:
plot2 <- 
data %>% 
  arrange(person_id, entry_datetime) %>% 
  group_by(person_id) %>% 
  slice_head(n = 1) %>%  
  ungroup() %>% 
  count(concept_id) %>% 
  full_join(concepts, by = "concept_id") %>% 
  mutate(concept_name = str_wrap(concept_name, width = 40)) %>% 
  mutate(n = ifelse(is.na(n), 0, n)) %>% 
  ggplot(aes(x = reorder(concept_name, n), y = n, fill = concept_name)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  labs(
    x = "Concept Name",
    y = "Unique Person Count",
    fill = "Concept ID"
  ) +
  theme_minimal() +
  scale_fill_manual(values = palette_21) +
  theme(
    legend.title = element_text(size = 16), 
    legend.text  = element_text(size = 14),  
    axis.text.x = element_text(size = 20), 
    axis.title.x = element_text(size = 20), 
    axis.text.y = element_text(size = 20), 
    axis.title.y = element_text(size = 20),
    legend.direction = "vertical",            # force vertical layout
    legend.key.height = unit(1.2, "cm"),      # taller keys
    legend.key.width  = unit(0.5, "cm"),
    legend.spacing.y  = unit(0.5, "cm")
  ) +
  guides(
    fill = guide_legend(ncol = 1)             # applies to fill aesthetic
  )


In [ ]:
plot <- (plot1 | plot2) + 
  patchwork::plot_annotation(tag_levels = "A") &
  theme(
    plot.title = element_text(size = 20),      # title size
    plot.subtitle = element_text(size = 16),                  # subtitle size
    plot.caption = element_text(size = 12),                   # caption size
    plot.tag = element_text(size = 18)         # tag size
  )


In [ ]:
ggsave("aml_ehr_count_plot.png", plot = plot, width = 40, height = 15, dpi = 300)

In [ ]:
plot1 <- 
data %>% 
  inner_join(concepts, by = "concept_id") %>% 
  mutate(concept_name = str_wrap(concept_name, width = 40)) %>% 
  ggplot(aes(x = age_at_event, fill = as.factor(concept_name))) + 
  geom_histogram(bins = 50, color = "white", alpha = 0.8) + 
  theme_bw() +
  scale_fill_manual(values = palette_21) + 
  labs(
    x = "Age at Electronic Health Record",
    y = "Count",
    fill = "Concept ID"
  ) + 
  guides(fill = "none")  +
  theme(
    legend.title = element_text(size = 16), 
    legend.text  = element_text(size = 14),  
    axis.text.x = element_text(size = 20), 
    axis.title.x = element_text(size = 20), 
    axis.text.y = element_text(size = 20), 
    axis.title.y = element_text(size = 20)
  )


In [ ]:
plot2 <- 
data %>% 
  inner_join(concepts, by = "concept_id") %>% 
  mutate(concept_name = str_wrap(concept_name, width = 40)) %>% 
  arrange(person_id, entry_datetime) %>% 
  group_by(concept_id, person_id) %>% 
  slice_head(n = 1) %>%  # keeps the first entry per person within each concept
  ungroup() %>% 
  ggplot(aes(x = age_at_event, fill = as.factor(concept_name))) + 
  geom_histogram(bins = 50, color = "white", alpha = 0.8) + 
  theme_bw() +
  scale_fill_manual(values = palette_21) + 
  labs(
    x = "Age at First Electronic Health Record",
    y = "Count",
    fill = "Concept ID"
  )  +
  theme(
    legend.title = element_text(size = 16), 
    legend.text  = element_text(size = 14),  
    axis.text.x = element_text(size = 20), 
    axis.title.x = element_text(size = 20), 
    axis.text.y = element_text(size = 20), 
    axis.title.y = element_text(size = 20)
  ) 

In [ ]:
plot <- (plot1 | plot2) + 
    patchwork::plot_annotation(tag_levels = "A") &
  theme(
    plot.title = element_text(size = 20),      # title size
    plot.subtitle = element_text(size = 16),                  # subtitle size
    plot.caption = element_text(size = 12),                   # caption size
    plot.tag = element_text(size = 18)         # tag size
  )


In [ ]:
ggsave("aml_ehr_age_plot.png", plot = plot, width = 30, height = 10, dpi = 300)